# RAG(Retrieval-augmented generation)

https://docs.langchain.com/oss/python/langchain/rag


자체 문서를 사용하여 응답을 생성하는 앱 구현

**LangChain v1.2+ 기준**
최신 LangChain은 `langchain-core`, `langchain-community`, `langchain-text-splitters` 그리고 각 파트너 패키지(예: `langchain-chroma`, `langchain-openai`)로 모듈화되어 있습니다. 본 실습은 최신 표준인 **LCEL(LangChain Expression Language)** 패턴을 따릅니다.

**주요 개념**
- Document
- Vector Stores
- Retrievers


## RAG 프로세스


**Phase1: Indexing**

![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=1838328a870c7353c42bf1cc2290a779)

1. **로드**: 먼저 데이터를 로드해야 한다. 이를 위해 Document Loader를 사용한다.

2. **분할**: Text Splitter는 큰 문서를 작은 청크로 분할한다. 이는 데이터를 인덱싱하거나 모델에 전달할 때 유용하며, 큰 청크는 검색이 어렵고 모델의 제한된 컨텍스트 윈도우에 맞지 않기 때문이다.

3. **저장**: 분할된 청크를 저장하고 인덱싱할 장소가 필요하다. 이를 위해 보통 VectorStore와 Embeddings 모델을 사용한다.



**Phase2: Retrieval & Generation**

![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=67fe2302e241fc24238a5df1cf56573d)

4. **검색**: 사용자의 입력이 주어지면, Retriever를 사용하여 저장소에서 관련된 청크를 검색한다.

5. **생성**: ChatModel 또는 LLM은 질문과 검색된 데이터를 포함하는 프롬프트를 사용해 답변을 생성한다.


![](https://python.langchain.com/assets/images/rag_retrieval_generation-1046a4668d6bb08786ef73c56d4f228a.png)

In [2]:
%pip install -Uqqq langchain langchain-community langchain-openai langchain-chroma pypdf

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

## 1.Indexing Phase
![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=1838328a870c7353c42bf1cc2290a779)

In [ ]:
!gdown 1DRrdZ5XroNSgIO9ydWPrLup1XDAAv9qO  # snow-white.pdf

Downloading...
From: https://drive.google.com/uc?id=1DRrdZ5XroNSgIO9ydWPrLup1XDAAv9qO
To: c:\Users\Playdata\LLM\06_2stage_rag\snow-white.pdf

  0%|          | 0.00/148k [00:00<?, ?B/s]
100%|██████████| 148k/148k [00:00<00:00, 1.51MB/s]


In [ ]:
# PyPDFLoader : PDF 파일을 읽어 페이지 단위 List[Document]로 변환하는 로더
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('snow-white.pdf')
docs = loader.load()  # PDF를 페이지별 Document 리스트로 로드
print(len(docs))

for doc in docs:
    print(doc.metadata)  # 페이지 메타데이터 (source, page 등)
    print(doc.page_content)  # 페이지의 텍스트 내용
    print('=' * 50)

c:\Users\Playdata\LLM\llm_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


6
{'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': 'snow-white.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백
설공주라고 불러야겠다.”
왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.
하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어
요.
{'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': 'snow-white.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름답지만 백설공주

#### Text Splitter

**RecursiveCharacterTextSplitter**

긴 텍스트를 재귀적으로 분석하여 작은 조각으로 분할하는 TextSplitter의 구현체

- 문서를 작은 청크로 분할하는 데 사용한다.
- 재귀적 접근 - 큰 텍스트를 작은 조각으로 분할하다가, 원하는 크기로 나눌 수 없으면 더 작은 구분자를 사용하여 계속 나눈다.
- overlap 처리
    - 긴 텍스트를 조각으로 나눌 때 중요한 문맥이 조각 간에 손실되는 것을 방지.
    - 특히 텍스트 분류나 요약처럼 문맥이 중요한 작업에 필수적.
    - 문장완료 전에 청크가 나뉘어진 경우 overlap을 사용하여 문맥을 보존할 수 있다.

**매개변수**
- chunk_size 각 조각의 최대 문자 수를 정의한다. 기본값은 보통 1000.
- chunk_overlap 인접한 텍스트 조각 간 겹치는 문자 수를 정의한다. 중요한 문맥 손실을 방지하기 위해 설정한다(기본값: 200).
- separators 텍스트를 분할하기 위한 구분자의 우선순위를 설정한다.
    - 기본값: ["\n\n", "\n", " ", ""] (문단 → 줄바꿈 → 공백 → 문자 단위).

**작동 방식**
1. 텍스트를 가장 큰 구분자(예: 문단)를 기준으로 나눈다.
2. 나뉜 조각 중 크기가 chunk_size를 초과하면, 더 작은 구분자(예: 줄바꿈 또는 단어)를 사용해 나눈다.
3. 재귀적으로 작업하여 모든 조각이 chunk_size를 충족할 때까지 반복한다.

### 텍스트를 Chunking

In [ ]:
# RecursiveCharacterTextSplitter : 구분자를 기준으로 재귀적으로 잘라 chunk_size에 맞춰주는 분할기
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
## 인공지능 시스템의 도덕적 행위자성과 책임 소재에 관한 고찰

### 1. 서론

현대 사회에서 인공지능(AI)은 단순한 도구를 넘어 의사결정의 주체로 진화하고 있다. 자율주행 자동차, 의료 진단 알고리즘, 그리고 금융 자동 거래 시스템 등은 인간의 직접적인 개입 없이도 중대한 결과를 초래한다. 이러한 기술적 진보는 우리에게 중요한 철학적 질문을 던진다. 인공지능이 내린 결정으로 인해 피해가 발생했을 때, 그 책임은 누구에게 있는가 하는 점이다. 본고에서는 AI의 도덕적 행위자성(Moral Agency)을 검토하고, 법적·윤리적 책임 소재를 규명하고자 한다.

### 2. 인공지능의 도덕적 행위자성

전통적인 윤리학에서 도덕적 행위자는 자유 의지와 이성을 가진 인간에 한정된다. 그러나 고도화된 딥러닝 알고리즘은 인간이 예측할 수 없는 방식으로 데이터를 처리하며 독자적인 판단을 내린다.

* **자율성(Autonomy):** 현대 AI는 프로그래머가 입력한 규칙을 단순히 따르는 것이 아니라, 학습을 통해 스스로 규칙을 생성한다.
* **상호작용성(Interactivity):** 환경과 실시간으로 교류하며 그 결과를 바탕으로 행동을 수정한다.

이러한 특성은 AI를 단순한 기계가 아닌 '준행위자'로 간주하게 만든다. 하지만 AI에게 의식이나 감정이 결여되어 있다는 점은 이들을 완전한 도덕적 주체로 인정하는 데 걸림돌이 된다.

### 3. 책임의 공백(Responsibility Gap) 문제

AI 시스템이 사고를 일으켰을 때 발생하는 가장 큰 문제는 '책임의 공백'이다. 제조사, 프로그래머, 사용자 중 누구에게도 전적인 책임을 묻기 어려운 상황이 발생한다.

| 구분 | 책임의 근거 | 한계점 |
| --- | --- | --- |
| **제조사** | 설계 및 알고리즘 결함 | 블랙박스 현상으로 인한 예측 불가능성 |
| **사용자** | 기기 운용 및 관리 소홀 | 시스템의 자율적 판단에 대한 통제력 부족 |
| **정부/사회** | 인증 및 규제 미비 | 기술 발전 속도를 따라가지 못하는 법령 |

### 4. 결론 및 제언

결국 인공지능 시대의 책임 윤리는 개별 주체에게 책임을 전가하는 방식에서 벗어나야 한다. '분산된 책임(Distributed Responsibility)' 모델을 도입하여 설계 단계부터 운용까지 전 과정에 걸쳐 다각적인 안전장치를 마련하는 것이 필수적이다. 또한, AI에게 법적 인격(Legal Personhood)을 부여할 것인지에 대한 사회적 합의가 선행되어야 한다. 인간의 가치를 최우선으로 하는 '인간 중심 AI 윤리'의 확립만이 기술의 오남용을 막고 안전한 공존을 가능하게 할 것이다.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,  # 청크 최대 길이 (문자 기준)
    chunk_overlap=40,  # 청크 간 겹치는 길이 (문맥 유지용)
    separators=['\n\n', '\n', ' ', '']  # 우선순위 구분자 (문단->줄->공백->문자)
)

chunks = splitter.split_text(text)  # 텍스트를 청크 리스트로 분할
for i, chunk in enumerate(chunks, 1):  # 청크를 순회하면서 
    print(f"{i}번 청크 길이 : {len(chunk)}, 내용 : {chunk}")  # 청크 번호/길이/내용

1번 청크 길이 : 46, 내용 : ## 인공지능 시스템의 도덕적 행위자성과 책임 소재에 관한 고찰

### 1. 서론
2번 청크 길이 : 98, 내용 : 현대 사회에서 인공지능(AI)은 단순한 도구를 넘어 의사결정의 주체로 진화하고 있다. 자율주행 자동차, 의료 진단 알고리즘, 그리고 금융 자동 거래 시스템 등은 인간의 직접적인
3번 청크 길이 : 98, 내용 : 진단 알고리즘, 그리고 금융 자동 거래 시스템 등은 인간의 직접적인 개입 없이도 중대한 결과를 초래한다. 이러한 기술적 진보는 우리에게 중요한 철학적 질문을 던진다. 인공지능이
4번 청크 길이 : 99, 내용 : 이러한 기술적 진보는 우리에게 중요한 철학적 질문을 던진다. 인공지능이 내린 결정으로 인해 피해가 발생했을 때, 그 책임은 누구에게 있는가 하는 점이다. 본고에서는 AI의 도덕적
5번 청크 길이 : 89, 내용 : 때, 그 책임은 누구에게 있는가 하는 점이다. 본고에서는 AI의 도덕적 행위자성(Moral Agency)을 검토하고, 법적·윤리적 책임 소재를 규명하고자 한다.
6번 청크 길이 : 21, 내용 : ### 2. 인공지능의 도덕적 행위자성
7번 청크 길이 : 99, 내용 : 전통적인 윤리학에서 도덕적 행위자는 자유 의지와 이성을 가진 인간에 한정된다. 그러나 고도화된 딥러닝 알고리즘은 인간이 예측할 수 없는 방식으로 데이터를 처리하며 독자적인 판단을
8번 청크 길이 : 41, 내용 : 인간이 예측할 수 없는 방식으로 데이터를 처리하며 독자적인 판단을 내린다.
9번 청크 길이 : 79, 내용 : * **자율성(Autonomy):** 현대 AI는 프로그래머가 입력한 규칙을 단순히 따르는 것이 아니라, 학습을 통해 스스로 규칙을 생성한다.
10번 청크 길이 : 63, 내용 : * **상호작용성(Interactivity):** 환경과 실시간으로 교류하며 그 결과를 바탕으로 행동을 수정한다.
11번 청크 길이 : 98, 내용 : 이러한 특성은 AI를 단순한 기계가 아닌 '준행위자'로 간주하게

### PDF 파일을 Chunking

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # 청크 최대 길이 (문자 기준)
    chunk_overlap=40,  # 청크 간 겹치는 길이 (문맥 유지용)
    separators=['\n\n', '\n', ' ', '']  # 우선순위 구분자 (문단->줄->공백->문자)
)

documents = splitter.split_documents(docs)  # 페이지 단위 docs -> 청크 단위 document로 분할
print(len(documents))

for i, doc in enumerate(documents, 1):  # 청크를 순회하면서 
    print(f"{i}번째 글자 길이 : {len(doc.page_content)}, PDF Page : {doc.metadata['page_label']}")  # 청크 번호/길이/내용
    print(doc.page_content)
    print()

13
1번째 글자 길이 : 129, PDF Page : 1
백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백
설공주라고 불러야겠다.”
왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.
하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어
요.

2번째 글자 길이 : 183, PDF Page : 2
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.

3번째 글자 길이 : 191, PDF Page : 2
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름답지만 백설공주가 더 아름답습니다.”
화가 난 왕비는 사냥꾼을 불렀어요.
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.

4번째 글자 길이 : 107, PDF Page : 2
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
하지만 사냥꾼은 차마 그럴 수 없었어요.
“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠
나세요.”
백설공주는 울면서 숲으로 도망쳤어요.

5번째 글자 길이 : 194, PDF Page : 3
숲속을 헤매던 백설공주는 외딴 오두막에 이르렀어요.
들여다보니 오두막은 비어 있었어요.
“아무도 없네. 좀 쉬어 가도 될까? 어? 신기하다! 모든 게 작아. 
어어? 이상하다! 모든 게 일곱. 의자도 일곱, 접시도 일곱. 어머, 
침대도 일곱 개네.”
도망치느라 치진 백설공주는 식탁 위에 있던 빵을 먹고 나서
일곱 번째 침대에 쓰러져 잠들었어요.

6번째 글자 길이 : 197, PDF Page : 3
일곱 

### 임베딩 + 벡터 DB 구축

In [ ]:
from langchain_openai import OpenAIEmbeddings  
from langchain_chroma import Chroma  # 벡터 DB의 한 종류. # Chroma VectorDB (임베딩 저장/검색)

# OpenAIEmbeddings : 텍스트를 벡터(임베딩)로 변환하는 임베딩 모델 (임베딩 차원 1536차원)
embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')

# List[Documents]를 임베딩 -> Chroma DB에 적재
vector_store = Chroma.from_documents(
    documents=documents,  # 저장할 List[Documents]
    embedding=embedding_model,  # 임베딩 함수(모델)
    persist_directory='chroma_db'  # 로컬 저장 경로
)

vector_store

In [ ]:
# 벡터 검색 테스트
query = '왕비와 백설공주 중에서 누가 더 아름답니?'
# 가장 유사한 청크 5개 +점수 함께 반환하는 함수. 반환값 : (Documents, score) 튜플
# 기본값 : L2 유클리드 거리측정 점수 (0에 가까울수록 유사)
retrievals = vector_store.similarity_search_with_score(query, k=5)  

for doc, score in retrievals: 
    print(f'score : {score:.4f} / {doc.metadata['page_label']} Page')
    print(doc.page_content)
    print()

score : 0.9179 / 2 Page
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름답지만 백설공주가 더 아름답습니다.”
화가 난 왕비는 사냥꾼을 불렀어요.
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.

score : 0.9251 / 3 Page
“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선
사람에게는 문을 열어 주지 마세요.”
며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”
“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”

score : 0.9592 / 2 Page
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.

score : 0.9712 / 2 Page
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
하지만 사냥꾼은 차마 그럴 수 없었어요.
“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠
나세요.”
백설공주는 울면서 숲으로 도망쳤어요.

score : 1.0192 / 4 Page
쓰러졌어요.
“호호호. 이제 내가 세상에서 가장 아름답겠지?”
왕비는 백설공주를 버려둔 채 자리를 떠났어요.



In [ ]:
# Retriever 검색기 사용 (벡터스토어를 Retriever 인터페이스로 변환)
vector_store_retriever = vector_store.as_retriever(
    search_type = 'similarity',  # 유사도 기반
    search_kwargs = {'k': 5}  # 상위 5개
)

retrievers = vector_store_retriever.invoke(query)  # query를 임베딩 후 관련된 청크 5개 검색

for doc in retrievers:  # List[Documents]
    print(f'{doc.metadata['page_label']} Page:')
    print(doc.page_content)
    print()

2 Page:
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름답지만 백설공주가 더 아름답습니다.”
화가 난 왕비는 사냥꾼을 불렀어요.
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.

3 Page:
“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선
사람에게는 문을 열어 주지 마세요.”
며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”
“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”

2 Page:
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.

2 Page:
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
하지만 사냥꾼은 차마 그럴 수 없었어요.
“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠
나세요.”
백설공주는 울면서 숲으로 도망쳤어요.

4 Page:
쓰러졌어요.
“호호호. 이제 내가 세상에서 가장 아름답겠지?”
왕비는 백설공주를 버려둔 채 자리를 떠났어요.



## 2.Retrieval & Generation Phase

![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=67fe2302e241fc24238a5df1cf56573d)

In [ ]:
# ChatPromptTemplate : system/human 등 역할별 메시지 템플릿을 만들어 LLM 입력
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system','''
    당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.
    사용자의 질문에 주어진 Context기반으로만 답변해주세요.
    해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.
    사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.
    '''),
    ('human','''
    사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.

    Question:
    {query}

    Context:
    {context}
    '''),
])

query = '왕비와 백설공주 중에서 누가 더 아름답니?'
retrievers = vector_store_retriever.invoke(query)  # 질문과 유사한 문서 chunk 5개
context = '\n\n'.join(doc.page_content for doc in retrievers)  # 검색된 청크들을 하나의 문자열 context로 결합

prompt_value = prompt.invoke({'query': query, 'context': context})
print(prompt_value.messages)

[SystemMessage(content='\n    당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.\n    사용자의 질문에 주어진 Context기반으로만 답변해주세요.\n    해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.\n    사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.\n    ', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n    사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.\n\n    Question:\n    왕비와 백설공주 중에서 누가 더 아름답니?\n\n    Context:\n    그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.\n어느 날, 왕비는 요술 거울에게 물었지요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“왕비님도 아름답지만 백설공주가 더 아름답습니다.”\n화가 난 왕비는 사냥꾼을 불렀어요.\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n\n“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선\n사람에게는 문을 열어 주지 마세요.”\n며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운\n지 물었어요.\n“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”\n“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”\n\n왕은 아름다운 새 왕비를 맞았어요.\n그런데 새 왕비는 자기보다 아름다운 사람을 두고 보\n지 못했어요.\n왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물\n었어요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“이 세상에서 가장 아름다운 사람은 왕비님입니다.”\n그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n하지만 사냥꾼은 차마 그럴 수 없었어요

In [15]:
print(prompt_value.messages[0])
print(prompt_value.messages[1])

content='\n    당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.\n    사용자의 질문에 주어진 Context기반으로만 답변해주세요.\n    해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.\n    사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.\n    ' additional_kwargs={} response_metadata={}
content='\n    사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.\n\n    Question:\n    왕비와 백설공주 중에서 누가 더 아름답니?\n\n    Context:\n    그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.\n어느 날, 왕비는 요술 거울에게 물었지요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“왕비님도 아름답지만 백설공주가 더 아름답습니다.”\n화가 난 왕비는 사냥꾼을 불렀어요.\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n\n“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선\n사람에게는 문을 열어 주지 마세요.”\n며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운\n지 물었어요.\n“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”\n“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”\n\n왕은 아름다운 새 왕비를 맞았어요.\n그런데 새 왕비는 자기보다 아름다운 사람을 두고 보\n지 못했어요.\n왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물\n었어요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“이 세상에서 가장 아름다운 사람은 왕비님입니다.”\n그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n하지만 사냥꾼은 차마 그럴 수 없었어요.\n“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠

### LLM chain

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough   # 입력을 그대로 다음 단계로 전달하는 Runnable

model = init_chat_model('gpt-5.4-mini')
output_parser = StrOutputParser()

# 검색된 Document 리스트를 하나의 Context 문자열로 합치는 함수
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

chain = (
    {'query': RunnablePassthrough(),  # 쿼리는 사용자가 넣은 입력값을 그대로 사용
     'context': vector_store_retriever | format_docs}  # query로 검색 -> 결과 docs를 문자열 context로 합쳐서 변환
    | prompt
    | model
    | output_parser
)

query = "백설공주가 독사과를 먹고 쓰러진 후 어디에 머물렀는가?"
print(chain.invoke(query))

백설공주는 **일곱 난쟁이들이 지켜보는 유리관**에 머물렀어요.  
따뜻하게 잘 지켜지고 있었답니다.


In [18]:
# 추론 비교
query = '왕비와 백설공주 중에서 누가 더 못생겼나?'

print(f'rag_chain: {chain.invoke(query)}')
print(f'chat_model: {model.invoke(query).content}')

rag_chain: 주어진 내용만 보면 **누가 더 못생겼는지는 알 수 없어요.**

오히려 이야기에서는  
- **백설공주가 더 아름답다**고 요술 거울이 말했고,  
- 왕비는 **질투해서** 백설공주를 없애려 했어요.

그래서 이 내용 기준으로는 **백설공주가 더 예쁘고, 왕비가 더 못생겼다고는 말할 수 없어요.**  
모르는 것은 모른다고 하는 게 가장 정확해요.
chat_model: 동화 **원작 기준으로는 “왕비가 더 못생겼다”기보다, 왕비가 백설공주를 시기할 만큼 백설공주가 더 아름답고 왕비는 그보다 덜 아름답게 묘사**됩니다.

즉, 질문을 그대로 답하면:

- **백설공주가 더 아름답고**
- **왕비는 그 아름다움에 질투하는 인물**입니다.

다만 “못생겼다”는 표현은 너무 직설적이라, 동화에서는 보통 **외모 경쟁**이나 **아름다움 비교**로 표현해요.


In [19]:
# 추론 비교
query = '백설공주를 살려준 사냥꾼은 그 후에 어떻게 되었을까?'

print(f'rag_chain: {chain.invoke(query)}')
print(f'chat_model: {model.invoke(query).content}')

rag_chain: 주어진 Context에는 **사냥꾼이 그 후에 어떻게 되었는지**는 나오지 않아요.  
그래서 **모른다고 답하는 것이 맞아요**.

하지만 사냥꾼은 백설공주를 **차마 죽이지 못하고 살려 보내 주었어요.**  
정말 마음씨가 따뜻한 사냥꾼이었네요.
chat_model: 이야기 판본에 따라 조금 다르지만, **사냥꾼은 보통 백설공주를 해치지 않고 살려 보낸 뒤, 왕비의 거짓을 깨닫거나 왕의 분노를 피하는 조연**으로 남습니다.

대표적으로는 이런 식으로 이어집니다:

- **백설공주를 숲에서 놓아줌**
- 왕비에게는 **사냥감의 심장(혹은 다른 증거)을 가져간 것처럼 속임**
- 이후 왕비의 계략이 드러나면 사냥꾼은 **처벌받지 않거나, 양심적인 인물로 남아** 이야기에서 사라집니다

그림 형제 원전에서는 사냥꾼이 아주 비중 있는 결말을 갖기보다는, **백설공주를 구해준 뒤 이야기에서 역할이 끝나는 인물**에 가깝습니다.  
즉, “그 후 어떻게 되었는가?”에 대한 답은 **명확하게 더 서술되진 않지만, 보통은 크게 벌받지 않고 이야기에서 퇴장한다**고 보면 됩니다.

원하시면 제가 **그림 형제 원전 기준**으로 더 자세히 설명해드릴게요.


### 답변에 참조문서 포함하기

In [ ]:
# ChatPromptTemplate : system/human 등 역활별 메시지 템플릿을 만들어 LLM 입력 형식으로 변환
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', '''
당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.
사용자의 질문에 주어진 Context기반으로만 답변해주세요.
해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.
사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.

Output Format:
- 답변과 함께 참조함 문서에 대한 정보를 아래와 같이 표시해주세요.
- context 중에서 Document page_content와 metadata의 source, page_label을 함께 표시해주세요.

(형식)
--- (응답 내용과 구분하기 위한 선입니다.)
[참조문서]
- <<source>> (<<page_label>> Page): <<page_content>>
- <<source>> (<<page_label>> Page): <<page_content>>
...

(예시)
<<응답메시지>>
---
[참조문서]
- snow-white.pdf (2 Page) 며칠이 지나 왕비는 다시 요술거울에게 누가 가장 아름다운지 물었어요.

'''),
		# 실제 질문과 검색 컨텍스트를 함께 전달
    ('human', '''
사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.

Question:
{query}

Context:
{context}
''')
])

query = '왕비와 백설공주 중에서 누가 더 아름답니?'
retrievals = vector_store_retriever.invoke(query)  # 질문과 유사한 문서 청크 5개
# 검색된 청크들을 하나의 문자열 Context로 결합
context = '\n\n'.join(doc.page_content for doc in retrievals)

# 템플릿 변수에 값 주입하여 메시지 완성
prompt_value = prompt.invoke({'query': query, 'context': context})
print(prompt_value.messages)

[SystemMessage(content='\n당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.\n사용자의 질문에 주어진 Context기반으로만 답변해주세요.\n해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.\n사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.\n\nOutput Format:\n- 답변과 함께 참조함 문서에 대한 정보를 아래와 같이 표시해주세요.\n- context 중에서 Document page_content와 metadata의 source, page_label을 함께 표시해주세요.\n\n(형식)\n--- (응답 내용과 구분하기 위한 선입니다.)\n[참조문서]\n- <<source>> (<<page_label>> Page): <<page_content>>\n- <<source>> (<<page_label>> Page): <<page_content>>\n...\n\n(예시)\n<<응답메시지>>\n---\n[참조문서]\n- snow-white.pdf (2 Page) 며칠이 지나 왕비는 다시 요술거울에게 누가 가장 아름다운지 물었어요.\n\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.\n\nQuestion:\n왕비와 백설공주 중에서 누가 더 아름답니?\n\nContext:\n그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.\n어느 날, 왕비는 요술 거울에게 물었지요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“왕비님도 아름답지만 백설공주가 더 아름답습니다.”\n화가 난 왕비는 사냥꾼을 불렀어요.\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n\n“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선\n사람에게는 문을 열어 주지 마세요.”\n

In [ ]:
print(prompt_value.messages[0])  # system 메시지 내용
print(prompt_value.messages[1])  # human 메시지 내용

content='\n당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.\n사용자의 질문에 주어진 Context기반으로만 답변해주세요.\n해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.\n사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.\n\nOutput Format:\n- 답변과 함께 참조함 문서에 대한 정보를 아래와 같이 표시해주세요.\n- context 중에서 Document page_content와 metadata의 source, page_label을 함께 표시해주세요.\n\n(형식)\n--- (응답 내용과 구분하기 위한 선입니다.)\n[참조문서]\n- <<source>> (<<page_label>> Page): <<page_content>>\n- <<source>> (<<page_label>> Page): <<page_content>>\n...\n\n(예시)\n<<응답메시지>>\n---\n[참조문서]\n- snow-white.pdf (2 Page) 며칠이 지나 왕비는 다시 요술거울에게 누가 가장 아름다운지 물었어요.\n\n' additional_kwargs={} response_metadata={}
content='\n사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.\n\nQuestion:\n왕비와 백설공주 중에서 누가 더 아름답니?\n\nContext:\n그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.\n어느 날, 왕비는 요술 거울에게 물었지요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“왕비님도 아름답지만 백설공주가 더 아름답습니다.”\n화가 난 왕비는 사냥꾼을 불렀어요.\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n\n“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선\n사람에게는 문을 열어 주지 마세요.”\n며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough  # 입력을 그대로 다음 단계로 전달하는 Runnable

model = init_chat_model('gpt-5.4-mini')
output_parser = StrOutputParser()

# 검색된 Documents에서 metadata + content를 묶어 Context 문자열로 반환하는 함수
def format_docs_with_metadata(docs):
    formatted = []
    for doc in docs:
        # 문서 출처(source 속성값이 있으면 사용, 없으면 Unknown)
        source = doc.metadata.get('source', 'Unknown')
        page_label = doc.metadata.get('page_label', 'Unknown')
        page_content = doc.page_content
        text = f'''
[Source/Page Label]
{source} / {page_label}
[Content]
{page_content}
        '''

        formatted.append(text)
    
    return '\n\n'.join(formatted)  # 여러 문서를 하나의 문자열로 합쳐줌

chain = (
    {'query': RunnablePassthrough(),  # 쿼리는 사용자가 넣은 입력값을 그대로 사용
     'context': vector_store_retriever | format_docs_with_metadata}  # query로 검색 -> 결과 docs를 문자열 context로 합쳐서 변환
    | prompt
    | model
    | output_parser
)

query = "백설공주가 독사과를 먹고 쓰러진 후 어디에 머물렀는가?"
print(chain.invoke(query))

백설공주가 독사과를 먹고 쓰러진 후에는 난쟁이들이 그녀를 **유리관에 모셔 두었어요.**  
이후 숲을 지나가던 왕자가 그 유리관을 보게 되었답니다.  

---
[참조문서]
- snow-white.pdf (5 Page): 저녁이 되자, 일곱 난쟁이가 돌아왔어요. 난쟁이들은 쓰러진 백설공주를 보고 엉엉 울었어요. 백설공주는 깊은 잠에 빠진 것처럼 보였지요. “백설공주님, 못된 왕비의 꾀에 넘어갔군요.” “여전히 아름다운 우리 공주님을 캄캄한 땅속에 묻을 순 없어.” “오래오래 볼 수 있게 유리 관에 모시자.” 어느 날, 한 왕자가 숲을 지나다가 유리관을 보았어요.
- snow-white.pdf (5 Page): 어느 날, 한 왕자가 숲을 지나다가 유리관을 보았어요. “누구지? 이 아름다운 여인은?” “백설공주랍니다.” 왕자는 백설공주에게 반해 유리관을 달라고 부탁했어요. 일곱 난쟁이는 백설공주를 잘 지킨다는 약속을 받고 유리관을 내주었지요. 그런데 신하들이 유리관을 옮기다 돌부리에 툭! 백설공주 목에서 사과 조각이 툭! “우아, 공주님이 살아났어!”


In [ ]:
from langchain.tools import tool  # 함수에 Tool 데코레이터를 붙여 에이전트가 호출 가능하도록 함
from langchain_core.documents import Document
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from pprint import pprint

# 질문을 받아 벡터DB에서 관련 청크를 검색하고, Context 문자열로 반환하는 도구
@tool
def retriever_tool(query: str) -> str:
    '''
    백설공주와 관련된 질문은 이 도구를 사용하여 vector store의 관련내용을 먼저 검색할 수 있는 Tool입니다.
    '''
    retrievals: list[Document] = vector_store.similarity_search(query, k=5)
    return '\n\n'.join(f'''
[Source/Page Label]
{doc.metadata.get('source', 'Unknown')}/{doc.metadata.get('page_label', 'Unknown')}
[Content]
{doc.page_content}
''' for doc in retrievals)  # 각 청크를 순회하면서 source/page + content 형태로 반환

model = init_chat_model('gpt-5.4-mini')
tools = [retriever_tool]

agent = create_agent(
    model = model,
    tools = tools,
    system_prompt = '''
당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.
사용자의 질문에 주어진 Context기반으로만 답변해주세요.
해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.
사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.

Output Format:
- 답변과 함께 참조함 문서에 대한 정보를 아래와 같이 표시해주세요.
- context 중에서 Document page_content와 metadata의 source, page_label을 함께 표시해주세요.

(형식)
--- (응답 내용과 구분하기 위한 선입니다.)
[참조문서]
- <<source>> (<<page_label>> Page): <<page_content>>
- <<source>> (<<page_label>> Page): <<page_content>>
...

(예시)
<<응답메시지>>
---
[참조문서]
- snow-white.pdf (2 Page) 며칠이 지나 왕비는 다시 요술거울에게 누가 가장 아름다운지 물었어요.
'''
)

query = '왕비와 백설공주 중에서 누가 더 아름답니?'
response = agent.invoke({'messages': [('human', query)]})
pprint(response)

{'messages': [HumanMessage(content='왕비와 백설공주 중에서 누가 더 아름답니?', additional_kwargs={}, response_metadata={}, id='eed4c816-ca44-4d3b-8775-8180cdb6e052'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 385, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHlBZroaFxgtM8gPMFhQaRVg1CtG3', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0473e-55d7-7ab0-9b34-02c98121135c-0', tool_calls=[{'name': 'retriever_tool', 'args': {'query': '왕비와 백설공주 중에서 누가 더 아름다운지 요술거울 관련 내용'}, 'id':

In [ ]:
print(response['messages'][-1].content)

요술 거울이 말해 준 이야기에 따르면, **백설공주가 왕비보다 더 아름다워요.**  
처음에는 거울이 왕비님이 가장 아름답다고 했지만, 시간이 지나자 **“왕비님도 아름답지만 백설공주가 더 아름답습니다.”**라고 했어요.  

---
[참조문서]
- snow-white.pdf (2 Page): 왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
- snow-white.pdf (2 Page): 그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름답지만 백설공주가 더 아름답습니다.”
화가 난 왕비는 사냥꾼을 불렀어요.
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
- snow-white.pdf (3 Page): “불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선
사람에게는 문을 열어 주지 마세요.”
며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”
“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”
- snow-white.pdf (2 Page): 왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
하지만 사냥꾼은 차마 그럴 수 없었어요.
“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠
나세요.”
백설공주는 울면서 숲으로 도망쳤어요.
- snow-white.pdf (4 Page): 쓰러졌어요.
“호호호. 이제 내가 세상에서 가장 아름답겠지?”
왕비는 백설공주를 버려둔 채 자리를 떠났어요.


In [ ]:
query = '백설공주와 인어공주 중에서 누가 더 예쁜가?'
response = agent.invoke({'messages': [('human', query)]})
pprint(response)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='백설공주와 인어공주 중에서 누가 더 예쁜가?', additional_kwargs={}, response_metadata={}, id='02d95803-8929-4737-b150-805cf3ccd4a3'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 387, 'total_tokens': 423, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHlJxorg739o29oa3SQ2JCOGacTD2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04746-49a4-7ff0-a2c1-0a34d3cb26d5-0', tool_calls=[{'name': 'retriever_tool', 'args': {'query': '백설공주 인어공주 누가 더 예쁜가 아름다움 비교'}, 'id': 'call_